<!-- AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06 -->
<!-- AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Write the root README and a Task 1 REFLECTION as decided in the grilling round', Date: 2026-10-07 -->

# Task 1: LLM-powered equity research assistant

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sathudeva7/CDAZZDEV-MLE-SATHURSAN/blob/main/task1_financial/task1_equity_research.ipynb)

This notebook is a thin driver. The logic lives in the `task1_financial/` modules
and in `common/` (the structured LLM helper and the Jev client), which carry
offline tests in `tests/`. Each section below matches one rubric row, says what it
shows, and names the file that does the work. Results are printed by code cells
only, so no number in the text can drift from the run.

| Rubric row | Section | Where the logic lives |
|---|---|---|
| 1A OHLCV data fetch | 1A.1 | `task1_financial/data.py` |
| 1A Indicator accuracy | 1A.2 | `task1_financial/indicators.py` |
| 1A News retrieval | 1A.3 | `task1_financial/news.py` |
| 1A Summary dictionary | 1A.4 | `task1_financial/summary.py`, `signals.py` |
| 1A Robustness | 1A.5 | every module: warnings instead of exceptions |
| 1B Per-headline JSON | 1B.1 | `task1_financial/sentiment.py`, `schemas.py` |
| 1B Signal reasoning quality | 1B.2 | `task1_financial/recommendation.py` |
| 1B Structured output validation | 1B.3 | `common/llm.py`, `task1_financial/schemas.py` |
| 1B Prompt engineering | 1B.4 | `task1_financial/prompts.py` |
| Bonus report rendering | Bonus | `task1_financial/report.py` |

**Models.** The submitted run uses the `free` LLM profile: Groq's free tier
(`openai/gpt-oss-120b`) with an OpenRouter free model as fallback. One deliberate
exception to the free-tier setup: **Jev** (TypeSafe's decision model, a paid API)
labels each headline's sentiment, because it returns measured probabilities
instead of a self-reported confidence. The LLM writes the reason and the
Recommendation (`docs/adr/0001-jev-decides-llm-explains.md`). Without
`JEV_API_KEY` the LLM labels every headline, so the notebook runs fully on free tiers.

**To run it yourself:** add `GROQ_API_KEY` (and optionally `OPENROUTER_API_KEY`
and `JEV_API_KEY`) under Colab's Secrets (the key icon), then Runtime → Run all.

## Setup

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
import os
import subprocess
import sys
import time
from pathlib import Path

RUN_STARTED = time.perf_counter()
TICKER = "AAPL"
REPO_URL = "https://github.com/sathudeva7/CDAZZDEV-MLE-SATHURSAN"
REPO_REF = "main"  # the branch or tag the Colab run clones
SUBMISSION_PROFILE = "free"  # set here, not from the shell, so a local paid_dev setting cannot leak in
KEY_NAMES = ["GROQ_API_KEY", "OPENROUTER_API_KEY", "JEV_API_KEY"]

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content") / REPO_URL.rsplit("/", 1)[-1]
    if ROOT.exists():
        subprocess.run(["git", "-C", str(ROOT), "pull", "--quiet"], check=True)
    else:
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    # Locally the notebook runs from task1_financial/, one level below the repo root.
    ROOT = Path.cwd().parent if Path.cwd().name == "task1_financial" else Path.cwd()
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
sys.path.insert(0, str(ROOT))
os.environ["LLM_PROFILE"] = SUBMISSION_PROFILE

from common.llm import enable_console_logging
from common.llm_config import active_profile, secret

enable_console_logging()
profile = active_profile()
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=False).stdout.strip()  # the hash is informational
print(f"Running in {'Colab' if IN_COLAB else 'a local kernel'}, repo at {commit or 'unknown commit'}")
print(f"LLM profile: {profile.name} (primary {profile.primary.name}: {profile.primary.model}; "
      f"fallback {profile.fallback.name + ': ' + profile.fallback.model if profile.fallback else 'none'})")
for name in KEY_NAMES:  # presence only, never the value
    print(f"  {'✓' if secret(name) else '✗'} {name}")

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
# One run, one log: the call log is cleared so the committed file matches this notebook's outputs exactly.
from task1_financial.pipeline import LOG_DIR

CALL_LOG = LOG_DIR / profile.log_file
CALL_LOG.unlink(missing_ok=True)
print("Every LLM and Jev request of this run is logged to", CALL_LOG.relative_to(ROOT))

## 1A.1 OHLCV data fetch

`run_market_data` fetches the daily bars with `yfinance`, adds the indicators,
the momentum signal, the fundamentals, the summary and the headlines. The
window is computed from today's date in New York (`LOOKBACK_YEARS = 2`), never
from a date string, and prices are split- and dividend-adjusted.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
import inspect
import json

import pandas as pd

from task1_financial import data
from task1_financial.pipeline import run_market_data

market = run_market_data(TICKER)
prices = market.prices
first, last = prices.index[0].date(), prices.index[-1].date()
print(f"{TICKER}: {len(prices)} daily bars from {first} to {last} ({(last - first).days / 365.25:.2f} years, LOOKBACK_YEARS = {data.LOOKBACK_YEARS})")
print(inspect.getsource(data.fetch_ohlcv).split('    warnings')[0])  # how the window is computed
prices[data.OHLCV_COLUMNS].tail()

## 1A.2 Indicators from first principles

Every indicator is written with pandas and numpy only, in
`task1_financial/indicators.py`. The conventions follow StockCharts' worked
examples, and `tests/test_indicators.py` checks them against those published
values to 1e-6:

- **SMA**: plain rolling mean.
- **EMA** (used by MACD): k = 2 / (span + 1), seeded with the SMA of the first `span` values.
- **RSI 14**: Wilder smoothing, avg = (previous avg × 13 + today) / 14, seeded with a plain mean; RSI = 100 − 100 / (1 + avg gain / avg loss).
- **MACD (12, 26, 9)**: EMA12 − EMA26; signal = EMA9 of MACD; histogram = MACD − signal.
- **Bollinger (20, 2)**: SMA20 ± 2 × the *population* standard deviation (ddof = 0, Bollinger's definition; pandas defaults to the sample one).

The source, as it runs:

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
from task1_financial import indicators

for function in (indicators.sma, indicators.ema, indicators.rsi, indicators.macd, indicators.bollinger, indicators._seeded_smoothing):
    print(inspect.getsource(function))

**Independent check.** The cell below recomputes each indicator for the latest bar
a second way, with plain numpy and explicit loops rather than pandas' rolling and
`ewm`, and asserts that the module's value matches.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
import numpy as np

TOLERANCE = 1e-8
close = prices["Close"].to_numpy(dtype=float)
latest = prices.iloc[-1]


def ema_loop(values, span):
    """EMA as an explicit loop: seed with the mean of the first `span` values, then y += k * (x - y)."""
    k = 2 / (span + 1)
    out = [values[:span].mean()]
    for value in values[span:]:
        out.append(out[-1] + k * (value - out[-1]))
    return np.array(out)  # out[0] belongs to bar span - 1


def wilder_rsi_loop(values, period):
    change = np.diff(values)
    gains, losses = np.clip(change, 0, None), np.clip(-change, 0, None)
    avg_gain, avg_loss = gains[:period].mean(), losses[:period].mean()
    for gain, loss in zip(gains[period:], losses[period:]):
        avg_gain = (avg_gain * (period - 1) + gain) / period
        avg_loss = (avg_loss * (period - 1) + loss) / period
    return 100 - 100 / (1 + avg_gain / avg_loss)


macd_line = ema_loop(close, indicators.MACD_FAST_SPAN)[indicators.MACD_SLOW_SPAN - indicators.MACD_FAST_SPAN:] - ema_loop(close, indicators.MACD_SLOW_SPAN)
signal_line = ema_loop(macd_line, indicators.MACD_SIGNAL_SPAN)
band = close[-indicators.BB_WINDOW:]

checks = {
    indicators.COL_SMA_SHORT: close[-indicators.SMA_SHORT_WINDOW:].mean(),
    indicators.COL_SMA_LONG: close[-indicators.SMA_LONG_WINDOW:].mean(),
    indicators.COL_RSI: wilder_rsi_loop(close, indicators.RSI_PERIOD),
    indicators.COL_MACD: macd_line[-1],
    indicators.COL_MACD_SIGNAL: signal_line[-1],
    indicators.COL_MACD_HIST: macd_line[-1] - signal_line[-1],
    indicators.COL_BB_MIDDLE: band.mean(),
    indicators.COL_BB_UPPER: band.mean() + indicators.BB_NUM_STD * band.std(ddof=0),
    indicators.COL_BB_LOWER: band.mean() - indicators.BB_NUM_STD * band.std(ddof=0),
}
rows = [{"indicator": col, "module": latest[col], "recomputed": value, "abs diff": abs(latest[col] - value)} for col, value in checks.items()]
check_table = pd.DataFrame(rows)
assert (check_table["abs diff"] < TOLERANCE).all(), check_table
assert 0 <= latest[indicators.COL_RSI] <= 100
print(f"All {len(checks)} values match to within {TOLERANCE:g} on {last}.")
check_table

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
from task1_financial.report import plot_price_chart

plot_price_chart(prices, TICKER)

## 1A.3 News headlines

`fetch_headlines` reads Yahoo Finance's RSS feed first, then tops up from Google
News RSS (`"{ticker} stock"`). It keeps the last 7 days, removes duplicate
titles (keeping Yahoo's copy), and asks for 15 so that ten survive even after
duplicates. Both feeds are free and need no key.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
from task1_financial.news import MIN_HEADLINES

print(f"{len(market.headlines)} headlines (minimum {MIN_HEADLINES}), newest first")
pd.DataFrame(
    [{"published (UTC)": h.published.strftime("%Y-%m-%d %H:%M"), "feed": h.feed, "publisher": h.publisher, "headline": h.title} for h in market.headlines]
)

## 1A.4 Summary dictionary

`build_summary` returns a validated `MarketSummary`, dumped to a plain dict.
The required fields come first below; the rest records how each was reached.
The momentum signal is rule-based: five votes (price vs SMA 50, SMA 50 vs
SMA 200, MACD vs signal, MACD histogram direction, RSI vs 50), summed from −5 to +5.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
REQUIRED_FIELDS = ["current_price", "high_52w", "low_52w", "pe_ratio", "ytd_return_pct"]
summary = market.summary
for field in REQUIRED_FIELDS:
    print(f"{field:>15}: {summary[field]}")
print(f"{'momentum':>15}: {summary['momentum']['label']} (score {summary['momentum']['score']:+d})")
print()
print(json.dumps(summary, indent=2, default=str))

## 1A.5 Robustness

Data problems never raise. Each fallback is logged and listed in
`summary['warnings']`, and missing values are `None`. Here the pipeline is given
a ticker that does not exist: it retries, gives up, and still returns a complete
result. Thresholds and windows are named constants, listed after it.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
bad = run_market_data("NOTATICKERZZ")
print("bars:", len(bad.prices), "| headlines:", len(bad.headlines), "| current_price:", bad.summary["current_price"])
print("momentum:", bad.summary["momentum"]["label"], "| unavailable votes:", bad.summary["momentum"]["unavailable"])
print("warnings:")
for warning in bad.summary["warnings"]:
    print(" -", warning)

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
# No magic numbers: every window, period and threshold is a named constant.
{name: value for name, value in vars(indicators).items() if name.isupper() and isinstance(value, (int, float))}

## Task 1B: run the analysis

`run_analysis` scores every headline (Jev decides, the LLM explains), averages
the Sentiment score, then asks the LLM for a Buy, Hold or Sell call. Every
request goes through `common/llm.py` or `common/jev.py`, which validate the
answer and log it.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
from task1_financial.pipeline import run_analysis

analysis = run_analysis(market)
print("analysis warnings:", analysis.warnings or "none")

## 1B.1 Per-headline JSON and the Sentiment score

Each headline becomes a `HeadlineSentiment` with the brief's four fields
(`headline`, `sentiment`, `confidence`, `brief_reason`), plus where the label came
from (`scored_by`), Jev's probabilities, and the headline's `score`.

**Aggregation.** A headline's score is p(positive) − p(negative) from Jev's
probabilities, from −1 to +1. A neutral-leaning headline therefore counts for
little, rather than being forced to ±1. When the LLM labelled the headline, the
score is sign × confidence. The **Sentiment score** is the mean over scored
headlines, labelled positive above +0.2, negative below −0.2, and neutral otherwise.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
print("One record, as JSON:")
print(analysis.headline_sentiments[0].model_dump_json(indent=2))

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
pd.set_option("display.max_colwidth", 140)
headline_table = pd.DataFrame([item.model_dump(exclude={"probabilities"}) for item in analysis.headline_sentiments])
headline_table

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
scores = [item.score for item in analysis.headline_sentiments if item.score is not None]
print(f"mean of {len(scores)} headline scores = {sum(scores) / len(scores):+.4f}" if scores else "no headline scored")
print(analysis.sentiment.model_dump_json(indent=2))

## 1B.2 Signal reasoning

The LLM is given **facts, not a verdict**. Each fact pairs a raw value with a
relationship computed in code: distance from each moving average, whether the
SMA gap is widening, which way RSI and the MACD histogram are moving, %B and the
52-week position. The rule-based momentum label is deliberately withheld, so the
model has to combine the indicators itself. The prompt asks that every sentence
connect at least two facts, and the schema enforces 3–5 sentences.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
recommendation = analysis.recommendation
print("Facts the LLM received:")
for fact in recommendation.facts:
    print(" -", fact)
print()
print(f"Rule-based momentum signal (not shown to the LLM): {summary['momentum']['label']} ({summary['momentum']['score']:+d})")
print(f"LLM provider: {recommendation.provider} | ok: {recommendation.ok}")
print(recommendation.value.model_dump_json(indent=2))

## 1B.3 Structured output validation

Every answer is validated against a Pydantic schema before use. A failure is
logged, sent back to the model once with the validation error (a repair retry),
then passed to the fallback provider. If every provider fails, the caller's
fallback value is returned with `ok=False`, so the pipeline still finishes.

First, the schema alone rejects a two-sentence justification and a single key factor:

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
from pydantic import ValidationError

from task1_financial.schemas import Recommendation

try:
    Recommendation(recommendation="Buy", justification="Too short. Only two sentences.", key_factors=["one factor"])
except ValidationError as error:
    print(error)

Next, the real `StructuredLLM` runs against a **stub client**: about 10 lines that
replay scripted answers through the same SDK method, with no network. It logs to
a temporary folder, so this run's call log stays clean. Demo 1 answers with
broken JSON, then a valid answer: it ends **repaired**. Demo 2 answers invalidly on
every attempt from every provider: it ends with the **fallback** Hold, and the run
carries on.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
import tempfile
from types import SimpleNamespace

from common.llm import StructuredLLM
from task1_financial.prompts import RECOMMENDATION
from task1_financial.recommendation import FALLBACK_RECOMMENDATION


class StubClient:
    """Stands in for the OpenAI SDK client: each request returns the next scripted answer text."""

    def __init__(self, *answers):
        self.answers = list(answers)
        self.chat = SimpleNamespace(completions=SimpleNamespace(with_raw_response=SimpleNamespace(create=self._create)))

    def _create(self, **request):
        completion = SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=self.answers.pop(0)))], usage=None)
        return SimpleNamespace(retries_taken=0, parse=lambda: completion)


VALID = Recommendation(
    recommendation="Hold",
    justification="Price is above the 200-day average but below the 50-day one. RSI is flat near 50 while MACD sits under its signal. Mixed trend and weak momentum argue for waiting.",
    key_factors=["Price between the two moving averages", "Flat RSI with MACD below signal"],
).model_dump_json()
BROKEN_JSON = '{"recommendation": "Buy", "justification": '
TWO_SENTENCES = '{"recommendation": "Buy", "justification": "Up. Very up.", "key_factors": ["a", "b"]}'
UNKNOWN_LABEL = VALID.replace('"Hold"', '"Strong Buy"')
demo_variables = {"company": "Demo Inc.", "ticker": "DEMO", "as_of": "today", "horizon_days": 90, "technical_facts": "-", "sentiment_facts": "-"}


def demo(name, scripts):
    with tempfile.TemporaryDirectory() as tmp:
        stubs = {provider: StubClient(*answers) for provider, answers in scripts.items()}
        llm = StructuredLLM(tmp, client_factory=lambda provider, _profile: stubs[provider.name])
        result = llm.call(RECOMMENDATION, demo_variables, Recommendation, fallback=FALLBACK_RECOMMENDATION)
        logged = json.loads((Path(tmp) / profile.log_file).read_text())
    print(f"{name} returned {result.value.recommendation} (ok={result.ok}). Its call-log line:")
    print(json.dumps({key: logged[key] for key in ("prompt", "outcome", "attempts", "errors", "raw_excerpt")}, indent=2))
    print()


providers = [p.name for p in (profile.primary, profile.fallback) if p is not None]
demo("Demo 1", {providers[0]: [BROKEN_JSON, VALID], **{name: [] for name in providers[1:]}})
demo("Demo 2", {name: [TWO_SENTENCES, UNKNOWN_LABEL] for name in providers})

Finally, the real run's call log: one JSON line per request with the outcome
(`ok`, `repaired` or `fallback`), the attempts, the SDK's own retries, latency and
tokens. The committed `task1_financial/logs/llm_calls.jsonl` is this file.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
calls = pd.read_json(CALL_LOG, lines=True)
print(f"{len(calls)} logged requests in {CALL_LOG.relative_to(ROOT)}")
calls.groupby(["prompt", "provider", "outcome"], dropna=False).agg(
    requests=("ts", "size"),
    attempts=("attempts", "sum"),
    sdk_retries=("sdk_retries", "sum"),
    mean_latency_ms=("latency_ms", "mean"),
    prompt_tokens=("prompt_tokens", "sum"),
    completion_tokens=("completion_tokens", "sum"),
).round(0)

## 1B.4 Prompt engineering

Prompts are module-level constants in `task1_financial/prompts.py`, never inline
strings. Each has a name and a version, recorded in the call log, plus a
**system** message (role, rules, an example) and a **user** template filled at
call time. Jev's question is a `JevQuestion` constant, defined in the same way.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
from task1_financial import prompts

for prompt in (prompts.HEADLINE_REASON, prompts.HEADLINE_SENTIMENT, prompts.RECOMMENDATION):
    print(f"=== {prompt.name} v{prompt.version} ===")
    print("[system]\n" + prompt.system)
    print("[user template]\n" + prompt.user)
    print()
question = prompts.HEADLINE_SENTIMENT_QUESTION
print(f"=== Jev: {question.name} v{question.version} ===")
print(question.instructions)
print(json.dumps(question.options, indent=2))
print("[state template]\n" + question.state)

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
# The roles each request is sent with: system first, then the filled user template.
example = prompts.RECOMMENDATION.messages({**demo_variables, "technical_facts": "\n".join(recommendation.facts[:2])})
[(message["role"], message["content"][:80] + "...") for message in example]

## Bonus: one-page research brief

`report.py` writes the brief as Markdown from template constants, then renders it
to styled HTML with the chart embedded. It contains the company snapshot, a
technical outlook built from the same computed facts the LLM saw (so no extra
model call), the news sentiment with the top three headlines by strength of
sentiment, the Recommendation, how the brief was made, data notes, and the risk
disclaimer. The saved files are committed under `task1_financial/reports/`.

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
import html
from datetime import datetime

from IPython.display import Image, display

from task1_financial.pipeline import MARKET_TIMEZONE
from task1_financial.report import build_report, save_report

report = build_report(market, analysis, generated_at=datetime.now(MARKET_TIMEZONE))
for path in save_report(report):
    print("saved", path.relative_to(ROOT))
# An iframe keeps the page's own CSS from restyling the notebook.
display({"text/html": f'<iframe srcdoc="{html.escape(report.html)}" style="width:100%;height:1500px;border:0"></iframe>'}, raw=True)

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
# The same chart as a plain PNG output, which GitHub's notebook viewer always shows.
Image(report.chart_png) if report.chart_png else print("no chart")

## Run summary

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
sentiment = analysis.sentiment
print(f"Ticker:          {summary['ticker']} ({summary['company_name']}), data as of {summary['as_of']}")
print(f"Recommendation:  {recommendation.value.recommendation} (ok={recommendation.ok}, by {recommendation.provider})")
print(f"Sentiment score: {sentiment.score:+.2f} ({sentiment.label})" if sentiment.score is not None else "Sentiment score: unavailable")
print(f"Headlines:       {len(market.headlines)} retrieved; labelled by Jev {sentiment.by_jev}, by the LLM {sentiment.by_llm}, failed {sentiment.failed}")
print(f"Warnings:        {len(summary['warnings'])} data, {len(analysis.warnings)} analysis")
print(f"Logged requests: {len(calls)} ({calls['outcome'].value_counts().to_dict()})")
print(f"Run time:        {time.perf_counter() - RUN_STARTED:.0f} s")

In [ ]:
# AI-ASSISTED: Claude (claude-opus-5-5), Prompt: 'Build the Task 1 notebook and the bonus report as designed in the grilling rounds', Date: 2026-10-06
# Colab only: download the call log and the report, which are committed beside this notebook.
import zipfile

if IN_COLAB:
    from google.colab import files

    bundle = Path("/content/task1_run_outputs.zip")
    with zipfile.ZipFile(bundle, "w") as archive:
        for path in [CALL_LOG, *sorted((ROOT / "task1_financial" / "reports").glob(f"{TICKER}_*"))]:
            archive.write(path, path.relative_to(ROOT))
        print("bundled:", archive.namelist())
    files.download(str(bundle))
else:
    print("Local run: the log and the report are already in the repo.")